# Bayesian Optimisation Capstone — Week 3 Queries

All changes from Week 2 are marked with `# CHANGED:` or `# NEW:` comments.

### Summary of changes from Week 2
| Change | Week 2 | Week 3 | Reason |
|---|---|---|---|
| kappa values | {1.5, 2.0, 2.5} | {1.0, 1.5, 2.0, 2.5} | Added 1.0 for consistently improving functions |
| F1 grid resolution | 100 | 200 | Finer sweep to avoid boundary corner artefacts |
| F1 grid bounds | linspace(0,1) | linspace(0.01, 0.99) | Prevent snapping to [0,0] or [1,1] corners |
| Week 2 data appended | No | Yes | Growing dataset |
| Seed | 99 | 777 | Avoid replicating prior candidate draws |

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.spatial.distance import cdist

# CHANGED: seed updated from 99 to 777
np.random.seed(777)
print('Libraries loaded.')

## 2. Load All Historical Data (Initial + Week 1 + Week 2)

**CHANGED from Week 2:** Week 2 queries and outputs are now appended alongside Week 1.

In [ ]:
DATA_PATH = './data/'

# Week 1 history — unchanged
week1_queries = {
    1: np.array([0.020584, 0.969910]),
    2: np.array([0.813958, 0.968718]),
    3: np.array([0.412118, 0.126161, 0.493019]),
    4: np.array([0.390847, 0.455494, 0.413276, 0.470071]),
    5: np.array([0.342735, 0.792788, 0.998424, 0.935381]),
    6: np.array([0.690974, 0.164681, 0.103053, 0.037619, 0.154198]),
    7: np.array([0.030065, 0.557667, 0.280418, 0.193751, 0.310746, 0.755561]),
    8: np.array([0.190497, 0.131345, 0.026522, 0.042568, 0.730959, 0.283019, 0.005482, 0.388835])
}
week1_outputs = {
    1: 1.966e-321,        2: -0.017659539446735338,
    3: -0.03594378093624019, 4: -1.3242014729441993,
    5: 2021.562348501267, 6: -1.8227163856485316,
    7: 1.3096626182809736, 8: 9.810010198309
}

# CHANGED: Week 2 queries and outputs added
week2_queries = {
    1: np.array([0.999999, 0.999999]),
    2: np.array([0.673666, 0.918591]),
    3: np.array([0.506031, 0.565092, 0.511916]),
    4: np.array([0.462210, 0.470559, 0.312877, 0.405903]),
    5: np.array([0.505785, 0.709925, 0.994838, 0.985391]),
    6: np.array([0.506031, 0.565092, 0.511916, 0.972186, 0.614903]),
    7: np.array([0.057378, 0.366665, 0.383868, 0.136267, 0.331203, 0.752374]),
    8: np.array([0.032926, 0.283604, 0.006343, 0.409730, 0.814350, 0.536043, 0.151336, 0.699855])
}
week2_outputs = {
    1: 1.517648729565899e-192, 2: 0.510292287604992,
    3: -0.010612319162868315,  4: -2.197214238202132,
    5: 2201.3475945869486,     6: -1.107214322524676,
    7: 1.9584243277849602,     8: 9.8526837633915
}

# Build full datasets
functions = {}
for i in range(1, 9):
    X = np.load(f'{DATA_PATH}function_{i}_initial_inputs.npy')
    y = np.load(f'{DATA_PATH}function_{i}_initial_outputs.npy')
    # CHANGED: now appending two rounds of history
    X = np.vstack([X, week1_queries[i], week2_queries[i]])
    y = np.append(y, [week1_outputs[i], week2_outputs[i]])
    functions[i] = {'X': X, 'y': y}

print(f"{'Fn':<5} {'n pts':<8} {'Best Y':<16} {'W2 Y':<16} {'W2 Improved?'}")
print('-' * 60)
for i in range(1, 9):
    y_init_best = np.load(f'{DATA_PATH}function_{i}_initial_outputs.npy').max()
    w2 = week2_outputs[i]
    best = functions[i]['y'].max()
    improved = w2 > max(y_init_best, week1_outputs[i])
    print(f"F{i:<4} {len(functions[i]['y']):<8} {best:<16.6f} {w2:<16.6f} {'YES' if improved else 'no'}")

## 3. Feature-Output Correlation Check

Unchanged from Week 2. Used as a sanity check on GP directions.

In [ ]:
print("Feature-output Pearson correlations:\n")
for func_id, d in functions.items():
    X, y = d['X'], d['y']
    corrs = [round(float(np.corrcoef(X[:, j], y)[0, 1]), 3) for j in range(X.shape[1])]
    labels = [f'x{j+1}:{c}' for j, c in enumerate(corrs)]
    print(f"F{func_id}: {', '.join(labels)}")

## 4. Core GP and Acquisition Functions

GP fitting unchanged. `max_distance_query` updated with finer resolution and bounded grid.

In [ ]:
def fit_gp(X, y):
    """Unchanged from Week 2."""
    gp = GaussianProcessRegressor(
        kernel=Matern(nu=2.5),
        n_restarts_optimizer=10,
        normalize_y=True
    )
    gp.fit(X, y)
    return gp


def suggest_next_query(X, y, n_dims, n_candidates=100000, kappa=2.0, seed=777):
    """
    CHANGED from Week 2: seed updated to 777.
    Everything else unchanged.
    """
    rng = np.random.default_rng(seed)
    gp = fit_gp(X, y)
    candidates = rng.uniform(0, 1, size=(n_candidates, n_dims))
    mu, sigma = gp.predict(candidates, return_std=True)
    ucb = mu + kappa * sigma
    best_idx = np.argmax(ucb)
    best_x = candidates[best_idx]
    mu_b, sig_b = gp.predict(best_x.reshape(1, -1), return_std=True)
    return best_x, float(mu_b.ravel()[0]), float(sig_b.ravel()[0]), float(ucb[best_idx])


def max_distance_query(X_obs, resolution=200):
    """
    CHANGED from Week 2:
      - resolution increased from 100 to 200 (finer sweep)
      - grid bounded to [0.01, 0.99] instead of [0, 1]
        to prevent the algorithm snapping to boundary corners
        which are rarely informative and caused a [0,0] artefact in Week 2
    """
    # CHANGED: linspace now starts at 0.01 and ends at 0.99
    x1 = np.linspace(0.01, 0.99, resolution)
    x2 = np.linspace(0.01, 0.99, resolution)
    X1, X2 = np.meshgrid(x1, x2)
    grid = np.column_stack([X1.ravel(), X2.ravel()])
    dists = cdist(grid, X_obs)
    min_dists = dists.min(axis=1)
    return grid[np.argmax(min_dists)]


def format_query(x):
    """Unchanged."""
    return '-'.join([f'{v:.6f}' for v in np.clip(x, 0.0, 0.999999)])


print('Functions defined.')

## 5. Week 3 Kappa Strategy

**CHANGED from Week 2:** kappa=1.0 introduced for consistently improving functions.

| kappa | Functions | Rationale |
|---|---|---|
| 1.0 | F5, F7, F8 | Consistent improvement across rounds — exploit hard |
| 1.5 | F3 | New best found this round — begin exploiting |
| 2.0 | F6 | Recovering but not at initial best — balanced |
| 2.5 | F2, F4 | Stalled or regressed — explore new regions |
| N/A | F1 | Max-distance sweep — no GP signal after 3 rounds |

In [ ]:
# CHANGED: kappa=1.0 added for F5, F7, F8
kappa_per_function = {
    1: None,  # max-distance
    2: 2.5,
    3: 1.5,
    4: 2.5,
    5: 1.0,   # CHANGED from 1.5
    6: 2.0,
    7: 1.0,   # CHANGED from 2.0
    8: 1.0,   # CHANGED from 1.5
}

strategy_notes = {
    1: "Max-distance grid sweep (resolution=200, bounded [0.01,0.99])",
    2: "GP-UCB kappa=2.5 (still below initial best)",
    3: "GP-UCB kappa=1.5 (new best -0.011, begin exploiting)",
    4: "GP-UCB kappa=2.5 (regressed in W2, explore new region)",
    5: "GP-UCB kappa=1.0 (consistent gains every round, exploit hard)",
    6: "GP-UCB kappa=2.0 (recovering but not at initial best)",
    7: "GP-UCB kappa=1.0 (new best 1.958 in W2, exploit hard)",
    8: "GP-UCB kappa=1.0 (new best 9.853 in W2, exploit hard)",
}
print('Kappa assignments set.')

## 6. Run Week 3 Queries

In [ ]:
results = {}

for func_id, d in functions.items():
    X, y = d['X'], d['y']
    n_dims = X.shape[1]
    kappa = kappa_per_function[func_id]

    if func_id == 1:
        best_x = max_distance_query(X)  # CHANGED: uses updated resolution/bounds
        gp = fit_gp(X, y)
        mu_b = float(gp.predict(best_x.reshape(1, -1)).ravel()[0])
        sig_b = float(gp.predict(best_x.reshape(1, -1), return_std=True)[1].ravel()[0])
    else:
        best_x, mu_b, sig_b, _ = suggest_next_query(X, y, n_dims, kappa=kappa)

    query_str = format_query(best_x)
    results[func_id] = {
        'query_string': query_str,
        'best_x': best_x,
        'current_best_y': float(y.max()),
        'predicted_mu': mu_b,
        'predicted_sigma': sig_b,
        'kappa': kappa,
        'strategy': strategy_notes[func_id],
    }

    print(f"F{func_id} ({n_dims}D) | n={len(y)} | Best Y: {y.max():.6f}")
    print(f"  Strategy : {strategy_notes[func_id]}")
    print(f"  GP mu={mu_b:.4f} | sigma={sig_b:.4f}")
    print(f"  Query    : {query_str}")
    print()

## 7. Submission Strings

In [ ]:
print('=' * 62)
print('WEEK 3 SUBMISSION STRINGS')
print('=' * 62)
for func_id, r in results.items():
    print(f'Function {func_id}: {r["query_string"]}')
print('=' * 62)

## 8. Progress Tracker — All Rounds

**NEW in Week 3:** A cumulative best-Y table across all rounds to track optimisation progress at a glance.

In [ ]:
# CHANGED: added progress tracker across all three rounds
init_bests = {i: np.load(f'{DATA_PATH}function_{i}_initial_outputs.npy').max() for i in range(1,9)}

rows = []
for i in range(1, 9):
    rows.append({
        'Function': f'F{i}',
        'Dims': functions[i]['X'].shape[1],
        'Init Best': round(init_bests[i], 4),
        'After W1':  round(max(init_bests[i], week1_outputs[i]), 4),
        'After W2':  round(max(init_bests[i], week1_outputs[i], week2_outputs[i]), 4),
        'Current Best': round(functions[i]['y'].max(), 6),
        'kappa W3': kappa_per_function[i],
        'W3 Query': results[i]['query_string']
    })

df = pd.DataFrame(rows).set_index('Function')
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 240)
print(df.to_string())

## 9. GP Surface Plots for 2D Functions (F1 and F2)

In [ ]:
def plot_2d_gp(func_id, X, y, query_x, kappa=2.0, resolution=80, label='Week 3 query'):
    x1 = np.linspace(0, 1, resolution)
    x2 = np.linspace(0, 1, resolution)
    X1, X2 = np.meshgrid(x1, x2)
    grid = np.column_stack([X1.ravel(), X2.ravel()])
    gp = fit_gp(X, y)
    mu, sigma = gp.predict(grid, return_std=True)
    ucb = mu + (kappa or 2.0) * sigma

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (data, title, cmap) in zip(axes, [
        (mu.reshape(resolution, resolution),    'GP Mean',        'RdYlGn'),
        (sigma.reshape(resolution, resolution), 'GP Uncertainty', 'Blues'),
        (ucb.reshape(resolution, resolution),   f'UCB (k={kappa})', 'plasma'),
    ]):
        im = ax.contourf(X1, X2, data, levels=30, cmap=cmap)
        plt.colorbar(im, ax=ax)
        # CHANGED: colour-code all three rounds of history
        n_init = len(np.load(f'{DATA_PATH}function_{func_id}_initial_inputs.npy'))
        ax.scatter(X[:n_init, 0],    X[:n_init, 1],    c='white',  s=50, edgecolors='k', linewidths=1, zorder=5, label='Initial')
        ax.scatter(X[n_init, 0],     X[n_init, 1],     c='yellow', s=80, marker='D', edgecolors='k', linewidths=1, zorder=6, label='W1 query')
        ax.scatter(X[n_init+1, 0],   X[n_init+1, 1],   c='orange', s=80, marker='s', edgecolors='k', linewidths=1, zorder=6, label='W2 query')
        ax.scatter(query_x[0], query_x[1], c='red', s=150, marker='*', edgecolors='k', linewidths=0.8, zorder=7, label=label)
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('x1'); ax.set_ylabel('x2')
        ax.legend(fontsize=7)

    plt.suptitle(f'Function {func_id} — Week 3 GP Surfaces', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'function_{func_id}_week3_gp.png', dpi=150, bbox_inches='tight')
    plt.show()


for func_id in [1, 2]:
    label = 'Max-dist (W3)' if func_id == 1 else 'GP-UCB (W3)'
    plot_2d_gp(func_id, functions[func_id]['X'], functions[func_id]['y'],
               results[func_id]['best_x'], kappa=kappa_per_function[func_id], label=label)

## 10. How to Update for Week 4

```python
week3_queries = {
    1: np.array([0.418744, 0.463065]),
    2: np.array([0.718071, 0.879689]),
    3: np.array([0.611094, 0.382817, 0.600071]),
    4: np.array([0.424666, 0.520010, 0.477146, 0.381999]),
    5: np.array([0.524778, 0.842229, 0.984007, 0.984075]),
    6: np.array([0.357049, 0.223854, 0.497338, 0.808843, 0.076793]),
    7: np.array([0.007196, 0.287182, 0.431297, 0.103298, 0.259892, 0.771925]),
    8: np.array([0.024544, 0.175956, 0.116596, 0.359046, 0.449942, 0.482790, 0.135292, 0.369405])
}
week3_outputs = {
    1: <value>, 2: <value>, 3: <value>, 4: <value>,
    5: <value>, 6: <value>, 7: <value>, 8: <value>
}

for i in range(1, 9):
    functions[i]['X'] = np.vstack([functions[i]['X'], week3_queries[i]])
    functions[i]['y'] = np.append(functions[i]['y'], week3_outputs[i])
```

### Kappa schedule going forward
| Round | Improving functions | Stalled functions | No-signal (F1) |
|---|---|---|---|
| W3 | 1.0 | 2.5 | max-distance |
| W4 | 0.5 | 2.0 | max-distance or random restart |
| W5+ | 0.5 | 1.5 | Consider random restart for F1 |